# PokerAI: Statistical Analysis

**Can a poker agent be profitable without being strategically robust?**

I compare Counterfactual Regret Minimisation (CFR and CFR+), tabular Q learning and Deep Q Networks (DQN) on Kuhn and Leduc Poker. My main interest is the difference between earning chips against particular opponents and learning a strategy that is difficult to exploit.

This notebook analyses the **saved results from the PokerAI experiments**. It does not retrain any agents. All payoffs are **net chips per hand**, averaged over both player positions unless stated otherwise.

### Main findings

- **CFR+ is the least exploitable Leduc agent**, with exploitability **0.00496**, compared with **0.09797** for CFR.
- **DQN trained against Random makes the most against Random**, earning **+2.2238 chips per hand**, but its exploitability remains high at **3.2833**.
- **Q learning trained against Mixed is the most robust reinforcement learning agent in this comparison**, with exploitability **2.3333**.
- **14 of 15** Monte Carlo confidence intervals contain the corresponding exact expected value. One interval misses, which is possible with 95% intervals even when the simulation is correctly specified.

These are results from the saved training runs. They are not claims that one algorithm will always outperform another across different seeds, training budgets or poker games.


## 1. Data and evaluation approach

The source files are committed under \`results/kuhn/\` and \`results/leduc/\` on the project's \`master\` branch. The notebook can be run from either the repository root or the \`notebooks/\` folder.

I use three complementary measurements:

- **Expected value (EV):** expected net profit when one fixed strategy faces another. The game's finite tree allows exact enumeration of all card deals and possible actions.
- **Exploitability:** the gain available to an optimal best response. A smaller value indicates a strategy closer to equilibrium.
- **Simulated EV:** realised average profit from randomly sampled games. This varies between sessions, unlike exact EV.

For a pair of strategies \(\pi_A\) and \(\pi_B\), the seat balanced payoff is

\[
EV(A,B)=\frac{1}{2}\left(\mathbb E[u_0(\pi_A,\pi_B)]+\mathbb E[u_1(\pi_B,\pi_A)]\right).
\]

The first term evaluates A as player 0. The second evaluates A as player 1. In a zero sum game, \(EV(A,B)=-EV(B,A)\).

**Important design limitation:** CFR iterations traverse a game tree, while Q learning and DQN learn from sampled hands. Their training budgets are not directly equivalent. Most reported reinforcement learning results are also from one seed, so I treat differences as observations about these saved policies, not general algorithm rankings.


In [ ]:
from pathlib import Path
import csv

import numpy as np
import matplotlib.pyplot as plt

# find the repository root regardless of where the notebook was opened
root = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "results" / "leduc" / "agent_summary.csv").is_file()),
    None
)
if root is None:
    raise FileNotFoundError("Open this notebook from PokerAI or PokerAI/notebooks")

# keep figure styling consistent with the rest of the project
plt.rcParams.update({
    "figure.facecolor": "lightsteelblue",
    "axes.facecolor": "lemonchiffon",
    "savefig.facecolor": "auto",
    "axes.titleweight": "bold",
    "axes.titlepad": 12
})

def load_rows(path):
    # this one helper is reused for each saved experiment table
    with (root / path).open(newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))

kuhn = load_rows("results/kuhn/kuhn_final_benchmark.csv")
leduc = load_rows("results/leduc/agent_summary.csv")
cfr_runs = load_rows("results/leduc/cfr_comparison.csv")
head_to_head = load_rows("results/leduc/head_to_head_matrix.csv")
mc_summary = load_rows("results/leduc/monte_carlo_summary.csv")
print(f"Loaded {len(kuhn)} Kuhn policies, {len(leduc)} Leduc policies "
      f"and {len(mc_summary)} simulated pairings.")


## 2. Kuhn Poker: Does minimising regret produce robust strategies?

Kuhn Poker is a useful starting point because its decision space is small and an equilibrium benchmark is available. I use the project's saved final benchmark to compare exploitability.

![Kuhn Poker exploitability comparison](../results/kuhn/kuhn_final_exploitability.png)

*Source: \`results/kuhn/kuhn_final_exploitability.png\`. This is an existing project figure, not a new simulation.*

| Saved Kuhn strategy | Exploitability |
| --- | ---: |
| Analytical Nash reference | ≈ 0 |
| CFR | 0.002318 |
| CFR+ | 0.00000963 |
| Q learning Random | 0.333333 |
| DQN Random Huber | 0.333333 |

**Interpretation.** The saved CFR+ strategy is much closer to the analytical Nash reference than the reinforcement learning policies. Q learning and DQN were rewarded for profitable play against their training opponents, not for minimising the payoff of an optimal adversary. That difference in objective matters more than whether a method uses a neural network.

These are the values in \`kuhn_final_benchmark.csv\`, which evaluates the saved policies. Other Kuhn convergence files cover different iteration checkpoints and should not be confused with these final policy values.


In [ ]:
# check the recorded values behind the Kuhn comparison
for name in ("Nash", "CFR", "CFR+", "Q-learning Random", "DQN Random Huber"):
    row = next(item for item in kuhn if item["agent"] == name)
    print(f"{name:20s}  exploitability = {float(row['exploitability']):.8f}")


## 3. Leduc Poker: How quickly does CFR+ reduce exploitability?

Leduc is a harder test. It introduces a second betting round, a public card and the option to raise, producing **288 information sets** in our implementation.

I compare CFR and CFR+ at the same number of full tree iterations. A logarithmic vertical axis makes improvements near zero easier to see.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for algorithm, colour, marker in (
    ("CFR", "#425e87", "o"),
    ("CFR+", "#7051a8", "s")
):
    rows = [row for row in cfr_runs if row["algorithm"] == algorithm]
    iterations = [int(row["iterations"]) for row in rows]
    exploitability = [float(row["exploitability"]) for row in rows]
    ax.plot(iterations, exploitability, marker=marker,
            label=algorithm, color=colour, linewidth=2)

ax.set_yscale("log")
ax.set_xlabel("Full tree iterations")
ax.set_ylabel("Exploitability (log scale)")
ax.set_title("Leduc Poker: CFR and CFR+ Convergence")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation.** Both methods improve as training proceeds, but CFR+ reduces exploitability faster over this run. After **200 iterations**, CFR reaches **0.09797**, while CFR+ reaches **0.00496**, roughly **20 times lower**.

This compares the algorithms at equal iteration counts, not equal computation time. CFR+ performs additional player specific traversals in the implementation, so the iteration and wall clock comparisons answer different questions.


## 4. Leduc Poker: Profitability against weak and strong opponents

My main question is whether earning more against Random also means playing more robustly.

The figure compares exact EV against a uniformly random opponent and against our saved CFR+ reference. Positive values indicate profit for the named agent. CFR+ is an **approximate equilibrium strategy**, not an independently known exact Nash solution.


In [ ]:
names = [row["agent"] for row in leduc]
ev_random = np.array([float(row["ev_vs_random"]) for row in leduc])
ev_cfr = np.array([float(row["ev_vs_cfr_plus"]) for row in leduc])
positions = np.arange(len(names))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(positions - width / 2, ev_random, height=width,
        label="Versus Random", color="#526e95")
ax.barh(positions + width / 2, ev_cfr, height=width,
        label="Versus CFR+", color="#8062a5")
ax.set_yticks(positions, names)
ax.invert_yaxis()
ax.axvline(0, color="#34465d", linewidth=1)
ax.set_xlabel("Exact expected profit (chips per hand)")
ax.set_title("Leduc Poker: Opponent Strength Changes Profitability")
ax.grid(axis="x", alpha=0.23)
ax.legend()
plt.tight_layout()
plt.show()


**Interpretation.** DQN Random earns the most against Random at **+2.2238 chips per hand**, narrowly ahead of Q learning Random at **+2.1122**. But DQN Random loses **0.5135 chips per hand** against CFR+.

Mixed training helps the tabular agent. Q learning Mixed loses only **0.1803** against CFR+ and has exploitability **2.3333**, compared with **2.7667** for DQN Mixed. These results suggest that exposure to stronger opponents can improve robustness, although the runs do not isolate opponent mixture as a causal effect across repeated seeds.

The distinction is important: **high EV against Random measures exploitation of a weak opponent, not closeness to equilibrium**.


## 5. What happens when the trained agents face each other?

An agent might perform well against the two references above but struggle against a different style of play. I therefore examine all **15 unique matchups** between our six saved Leduc policies.

Each heatmap cell shows the exact, seat balanced EV of the **row agent** against the **column opponent**. Since poker is zero sum, the matrix should be antisymmetric.


In [ ]:
names = [row["agent"] for row in head_to_head]
matrix = np.array([
    [float(row[opponent]) for opponent in names]
    for row in head_to_head
])

# a quick consistency check on the stored matrix
assert np.allclose(matrix, -matrix.T, atol=1e-10)

fig, ax = plt.subplots(figsize=(10, 7))
limit = np.abs(matrix).max()
image = ax.imshow(matrix, cmap="PuOr", vmin=-limit, vmax=limit)
ax.set_xticks(np.arange(len(names)), names, rotation=35, ha="right")
ax.set_yticks(np.arange(len(names)), names)
ax.set_xlabel("Opponent")
ax.set_ylabel("Agent")
ax.set_title("Leduc Poker: Exact Head to Head EV")

for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, f"{matrix[i, j]:+.2f}", ha="center",
                va="center", fontsize=9,
                color="white" if abs(matrix[i, j]) > 0.65 * limit else "black")

fig.colorbar(image, ax=ax, label="Chips per hand")
plt.tight_layout()
plt.show()


**Interpretation.** CFR+ wins every direct comparison with the other five policies, including a small **+0.0226** advantage against CFR. Q learning Mixed is the strongest reinforcement learning policy in its direct comparisons: it beats both DQN policies and Q learning Random.

CFR nonetheless has a slightly higher **mean EV against this particular opponent pool** than CFR+ (**+0.4052 versus +0.3512**). That does not contradict CFR+ having lower exploitability. The tournament average depends on which agents were included and how they were weighted. It is not a measure of Nash convergence.


## 6. Monte Carlo validation: How reliable is a finite match?

Exact EV describes the long run expectation for two fixed strategies. A real match is finite, so realised profit can be different even when the strategies are unchanged.

I simulated **20 independent sessions of 1,000 hands for each of 15 matchups**, giving **300,000 hands in total**. Each session used 500 hands from each seat, with the same deal reused when positions were swapped. The **session average**, rather than each individual hand, is the independent observation for uncertainty calculations.

For session means \(\hat\mu_1,\ldots,\hat\mu_{20}\), I use

\[
\bar\mu = \frac{1}{20}\sum_{s=1}^{20}\hat\mu_s,\qquad
SE(\bar\mu) = \frac{s_{\mathrm{session}}}{\sqrt{20}},
\]

and an approximate 95% Student \(t\) interval

\[
\bar\mu \pm t_{0.975,19}\, SE(\bar\mu).
\]

This interval describes uncertainty in the *mean across sessions*, not the range of outcomes likely for one 1,000 hand session.

### Exact EV against simulated EV

![Leduc Monte Carlo confidence intervals](../results/leduc/monte_carlo_comparison.png)

*Source: \`results/leduc/monte_carlo_comparison.png\`. Circles show simulated means with 95% confidence intervals. Crosses show exact EV.*

### Why short sessions can give different answers

![CFR versus CFR+ session variation](../results/leduc/monte_carlo_cfr_variance.png)

*Source: \`results/leduc/monte_carlo_cfr_variance.png\`. The shaded band is a confidence interval for the mean, not a prediction interval for one session.*


In [ ]:
# check how frequently the 95 percent intervals contain exact EV
contained = sum(row["exact_ev_inside_ci"] == "True" for row in mc_summary)
missed = [
    row for row in mc_summary
    if row["exact_ev_inside_ci"] != "True"
]
errors = [float(row["absolute_error"]) for row in mc_summary]

print(f"Exact EV covered: {contained}/{len(mc_summary)} matchups")
print(f"Mean absolute difference: {np.mean(errors):.4f} chips per hand")
for row in missed:
    print(f"Outside interval: {row['agent_a']} vs {row['agent_b']}")
    print(f"Exact {float(row['exact_ev']):+.4f}, "
          f"simulated {float(row['simulated_ev']):+.4f}, "
          f"95% CI [{float(row['ci_lower_95']):+.4f}, "
          f"{float(row['ci_upper_95']):+.4f}]")


**Interpretation.** The exact EV falls inside the estimated 95% confidence interval for **14 of 15** matchups. The exception is **DQN Random versus DQN Mixed**: exact EV is **+0.1833**, while simulated EV is **+0.2309** with a 95% interval of **[+0.1905, +0.2713]**. One miss is not surprising across 15 intervals, but it is still worth reporting.

The close CFR versus CFR+ matchup illustrates why individual sessions can be misleading. The exact EV from CFR's perspective is **−0.0226**, while the simulated average is **−0.0168**. Its confidence interval **[−0.0428, +0.0092]** crosses zero, even though the exact evaluator determines a small negative expectation.

The simulations are a useful independent check on the gameplay and evaluation logic. They do not make the exact results more accurate and they do not quantify variation between independently retrained models.


## 7. Conclusions and limitations

The same basic lesson appears in both games. **An agent can earn a high payoff against an exploitable opponent and still be easy for a strong opponent to beat.** CFR and CFR+ target equilibrium robustness, while Q learning and DQN maximise rewards under their particular training conditions.

In these saved Leduc runs:

1. CFR+ is the closest to equilibrium, with exploitability **0.00496**.
2. DQN Random exploits Random most strongly, earning **+2.2238 chips per hand**, but this is not evidence of a robust strategy.
3. Q learning Mixed is the strongest reinforcement learning policy by exploitability and its direct matchups with other reinforcement learning agents.
4. The Monte Carlo experiment shows how realised session profits can differ from exact EV and why uncertainty matters in finite samples.

**Limitations.** Leduc remains a small poker game. The training comparisons use different computational budgets and mostly a single random seed. The direct tournament includes just six chosen policies. The Monte Carlo confidence intervals describe gameplay variability for those fixed policies, not the variability that would arise from retraining them.

For this project, I find the most valuable result is not that one algorithm always wins. It is that **opponent exploitation, equilibrium robustness and statistical uncertainty answer different questions**, and a good evaluation needs to keep them separate.

---

**Source data:** \`results/kuhn/kuhn_final_benchmark.csv\`, \`results/leduc/agent_summary.csv\`, \`results/leduc/cfr_comparison.csv\`, \`results/leduc/head_to_head_matrix.csv\` and \`results/leduc/monte_carlo_summary.csv\`. All are precomputed project results committed on \`master\`.
